# Teste do Pipeline Multi-Source — Noticias de Hoje

Este notebook testa a coleta de noticias de ITUB4 de **todas as 3 fontes** disponiveis:
1. **InfoMoney** — artigos completos via WordPress API
2. **CVM Fatos Relevantes** — comunicados oficiais
3. **Google News** — agregador (Valor, Reuters, Bloomberg, Exame, etc.)

Apos coletar, roda o FinBERT-PT-BR para analise de sentimento e gera as mesmas metricas do pipeline existente.

In [ ]:
import sys
import os
import json
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

sys.path.insert(0, '.')
sys.path.insert(0, '..')
sys.path.insert(0, '../1.news')

print(f"Data de hoje: {datetime.now().strftime('%Y-%m-%d')}")
print("Modulos carregados")

## 1. Coleta de noticias — InfoMoney

Usa o mesmo extrator do Stage 1 para buscar artigos recentes sobre ITUB4.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

def collect_infomoney_recent(ticker="ITUB4", max_articles=20):
    """Coleta artigos recentes do InfoMoney via WordPress REST API."""
    url = "https://www.infomoney.com.br/wp-json/wp/v2/posts"
    params = {
        "search": ticker,
        "per_page": max_articles,
        "orderby": "date",
        "order": "desc",
    }
    
    try:
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        posts = resp.json()
    except Exception as e:
        print(f"Erro ao acessar InfoMoney: {e}")
        return []
    
    articles = []
    for post in posts:
        # Limpar HTML do conteudo
        title = BeautifulSoup(post.get("title", {}).get("rendered", ""), "html.parser").get_text()
        excerpt = BeautifulSoup(post.get("excerpt", {}).get("rendered", ""), "html.parser").get_text()
        content = BeautifulSoup(post.get("content", {}).get("rendered", ""), "html.parser").get_text()
        
        articles.append({
            "date": post.get("date", "")[:10],
            "title": title.strip(),
            "text": f"{title}. {excerpt}. {content}".strip()[:5000],
            "source": "InfoMoney",
            "link": post.get("link", ""),
        })
    
    print(f"InfoMoney: {len(articles)} artigos coletados")
    return articles

infomoney_articles = collect_infomoney_recent("ITUB4", max_articles=20)
for a in infomoney_articles[:3]:
    print(f"  [{a['date']}] {a['title'][:80]}...")

## 2. Coleta de noticias — CVM Fatos Relevantes

Busca comunicados oficiais do Itau na CVM (ano atual).

In [ ]:
from cvm_collector import download_year, filter_itau_fatos, fetch_document_text

current_year = datetime.now().year
print(f"Buscando Fatos Relevantes de {current_year}...")

df_cvm = download_year(current_year)
df_fatos = filter_itau_fatos(df_cvm)

cvm_articles = []
for _, row in df_fatos.iterrows():
    date_str = row.get("Data_Referencia", "") or row.get("Data_Entrega", "")
    try:
        date_iso = pd.to_datetime(date_str).strftime("%Y-%m-%d")
    except:
        date_iso = str(date_str)
    
    link = row.get("Link_Download", "")
    text = fetch_document_text(link) if link else ""
    
    cvm_articles.append({
        "date": date_iso,
        "title": f"Fato Relevante - {row.get('Categoria', '')}",
        "text": text[:5000] if text else "",
        "source": "CVM",
        "link": link,
    })

print(f"CVM: {len(cvm_articles)} fatos relevantes coletados")
for a in cvm_articles[:3]:
    print(f"  [{a['date']}] {a['title'][:80]}...")

## 3. Coleta de noticias — Google News

Busca noticias recentes sobre ITUB4 de multiplas fontes via Google News.

In [ ]:
from google_news_collector import collect_gnews, collect_rss, fetch_full_text

# Coletar via gnews (periodo curto para teste)
gn_articles_raw = []
for query in ["ITUB4", "Itau Unibanco"]:
    try:
        arts = collect_gnews(query, period="7d", max_results=20)
        gn_articles_raw.extend(arts)
    except Exception as e:
        print(f"Erro gnews '{query}': {e}")
    
    try:
        arts = collect_rss(query)
        gn_articles_raw.extend(arts)
    except Exception as e:
        print(f"Erro RSS '{query}': {e}")

# Deduplicar por URL
seen = set()
gn_articles = []
for a in gn_articles_raw:
    if a["link"] not in seen:
        seen.add(a["link"])
        # Usar title + description como texto (full text e lento)
        text = f"{a['title']}. {a.get('description', '')}"
        gn_articles.append({
            "date": a["date"],
            "title": a["title"],
            "text": text.strip()[:5000],
            "source": f"Google News ({a.get('publisher', a.get('source', 'N/A'))})",
            "link": a["link"],
        })

print(f"Google News: {len(gn_articles)} artigos unicos coletados")
for a in gn_articles[:3]:
    print(f"  [{a['date']}] [{a['source']}] {a['title'][:60]}...")

## 4. Combinar todas as fontes

Junta todos os artigos e mostra um resumo por fonte.

In [ ]:
all_articles = infomoney_articles + cvm_articles + gn_articles

print(f"
{'='*60}")
print(f"TOTAL: {len(all_articles)} artigos de todas as fontes")
print(f"{'='*60}")
print(f"  InfoMoney:    {len(infomoney_articles)}")
print(f"  CVM:          {len(cvm_articles)}")
print(f"  Google News:  {len(gn_articles)}")

# Resumo por data
df_all = pd.DataFrame(all_articles)
df_all['date'] = pd.to_datetime(df_all['date'], errors='coerce')
print(f"
Periodo: {df_all['date'].min().date()} a {df_all['date'].max().date()}")
print(f"
Artigos por fonte:")
display(df_all['source'].value_counts().head(10))

## 5. Analise de sentimento — FinBERT-PT-BR

Roda o mesmo modelo FinBERT usado nos stages anteriores em TODOS os artigos coletados.

In [ ]:
import torch
from transformers import AutoTokenizer, BertForSequenceClassification

model_dir = '../4.finbert-br/FinBERT-PT-BR'
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = BertForSequenceClassification.from_pretrained(model_dir)
model.eval()

LABEL_MAP = {0: 'POSITIVE', 1: 'NEGATIVE', 2: 'NEUTRAL'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f'FinBERT carregado no device: {device}')

In [ ]:
def predict_batch(texts, batch_size=16):
    """Analisa sentimento de um lote de textos com FinBERT."""
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        # Truncar textos muito longos
        batch = [t[:512] if t else "" for t in batch]
        
        tokens = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=512).to(device)
        with torch.no_grad():
            logits = model(**tokens).logits.cpu().numpy()
        
        for log in logits:
            cls = int(np.argmax(log))
            results.append({
                'sentiment_class': cls,
                'sentiment': LABEL_MAP[cls],
                'logits': log.tolist(),
            })
    return results

# Preparar textos (titulo + texto disponivel)
texts = []
for a in all_articles:
    t = a.get('text', '') or a.get('title', '')
    texts.append(t[:1000])  # limitar para nao estourar memoria

print(f"Analisando sentimento de {len(texts)} artigos...")
sentiments = predict_batch(texts)

# Adicionar sentimento aos artigos
for art, sent in zip(all_articles, sentiments):
    art['sentiment_class'] = sent['sentiment_class']
    art['sentiment'] = sent['sentiment']
    art['sentiment_logits'] = sent['logits']

# Resumo
from collections import Counter
sent_counts = Counter(s['sentiment'] for s in sentiments)
print(f"
Distribuicao de sentimento:")
for label, count in sent_counts.most_common():
    print(f"  {label}: {count} ({count/len(sentiments)*100:.1f}%)")

## 6. Agregacao diaria de sentimento

Gera as mesmas metricas do pipeline existente:
- : quantidade de artigos no dia
- : media do logit positivo
- : media do logit negativo  
- : media do logit neutro
- : media da classe de sentimento (0=pos, 1=neg, 2=neu)

Estas sao as features que alimentam os modelos de previsao.

In [ ]:
# Criar DataFrame com sentimento por artigo
df_sent = pd.DataFrame([{
    'date': a['date'],
    'source': a['source'],
    'sentiment_class': a['sentiment_class'],
    'logit_pos': a['sentiment_logits'][0],
    'logit_neg': a['sentiment_logits'][1],
    'logit_neu': a['sentiment_logits'][2],
} for a in all_articles])

df_sent['date'] = pd.to_datetime(df_sent['date'], errors='coerce')
df_sent = df_sent.dropna(subset=['date'])

# Agregacao diaria (mesma logica do Stage 6)
daily = df_sent.groupby('date').agg(
    n_articles=('sentiment_class', 'count'),
    mean_logit_pos=('logit_pos', 'mean'),
    mean_logit_neg=('logit_neg', 'mean'),
    mean_logit_neu=('logit_neu', 'mean'),
    mean_sentiment=('sentiment_class', 'mean'),
).sort_index()

print(f"Sentimento diario agregado: {len(daily)} dias")
print(f"
Ultimos 10 dias:")
display(daily.tail(10))

## 7. Comparacao: sentimento por fonte

Compara o sentimento medio de cada fonte para ver se ha diferencas sistematicas.

In [ ]:
# Agregacao por fonte
source_summary = df_sent.groupby('source').agg(
    n_articles=('sentiment_class', 'count'),
    mean_logit_pos=('logit_pos', 'mean'),
    mean_logit_neg=('logit_neg', 'mean'),
    mean_logit_neu=('logit_neu', 'mean'),
    mean_sentiment=('sentiment_class', 'mean'),
    pct_positive=('sentiment_class', lambda x: (x == 0).mean()),
    pct_negative=('sentiment_class', lambda x: (x == 1).mean()),
    pct_neutral=('sentiment_class', lambda x: (x == 2).mean()),
).round(3)

print("Sentimento medio por fonte:")
display(source_summary)

## 8. Visualizacao

Graficos do sentimento coletado.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Pipeline Multi-Source — ITUB4 ({datetime.now().strftime("%Y-%m-%d")})', 
             fontsize=14, fontweight='bold')

# 1. Artigos por fonte
ax = axes[0, 0]
source_counts = df_sent['source'].value_counts()
# Agrupar fontes pequenas
top_sources = source_counts.head(8)
if len(source_counts) > 8:
    top_sources['Outros'] = source_counts.iloc[8:].sum()
top_sources.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Artigos por fonte')
ax.set_xlabel('Quantidade')

# 2. Sentimento por dia
ax = axes[0, 1]
if len(daily) > 1:
    ax.plot(daily.index, daily['mean_logit_pos'], 'g-', label='Positivo', alpha=0.7)
    ax.plot(daily.index, daily['mean_logit_neg'], 'r-', label='Negativo', alpha=0.7)
    ax.plot(daily.index, daily['mean_logit_neu'], 'b-', label='Neutro', alpha=0.7)
    ax.legend()
    ax.set_title('Logits medios por dia')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
else:
    ax.bar(['Positivo', 'Negativo', 'Neutro'], 
           [daily['mean_logit_pos'].iloc[0], daily['mean_logit_neg'].iloc[0], daily['mean_logit_neu'].iloc[0]],
           color=['green', 'red', 'blue'])
    ax.set_title('Logits medios (dia unico)')

# 3. Distribuicao de sentimento
ax = axes[1, 0]
sent_dist = df_sent['sentiment_class'].value_counts().sort_index()
labels = ['Positivo', 'Negativo', 'Neutro']
colors = ['#2ecc71', '#e74c3c', '#3498db']
bars = ax.bar([labels[i] for i in sent_dist.index], sent_dist.values, color=[colors[i] for i in sent_dist.index])
ax.set_title('Distribuicao de sentimento (todos os artigos)')
ax.set_ylabel('Quantidade')
for bar, val in zip(bars, sent_dist.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(val), ha='center')

# 4. Artigos por dia
ax = axes[1, 1]
if len(daily) > 1:
    ax.bar(daily.index, daily['n_articles'], color='steelblue', alpha=0.7)
    ax.set_title('Volume de noticias por dia')
    ax.set_ylabel('Artigos')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
else:
    ax.bar(['Hoje'], [daily['n_articles'].iloc[0]], color='steelblue')
    ax.set_title(f'Artigos hoje: {daily["n_articles"].iloc[0]}')

plt.tight_layout()
plt.savefig('results/test_pipeline_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Formato de saida — compativel com o pipeline existente

O DataFrame  tem o mesmo formato usado nos Stages 4 e 6 como input dos modelos.
Pode ser salvo como CSV e usado diretamente no Stage 7 para retreinar modelos com dados multi-source.

In [ ]:
# Salvar sentimento diario
daily.to_csv('multi_source_daily_sentiment.csv')
print(f"Sentimento diario salvo em multi_source_daily_sentiment.csv")
print(f"Formato: {daily.columns.tolist()}")
print(f"Shape: {daily.shape}")

# Salvar artigos completos com sentimento
with open('all_articles_with_sentiment.json', 'w', encoding='utf-8') as f:
    # Converter datas para string para JSON
    for a in all_articles:
        if isinstance(a.get('date'), pd.Timestamp):
            a['date'] = a['date'].strftime('%Y-%m-%d')
    json.dump(all_articles, f, ensure_ascii=False, indent=2, default=str)

print(f"Artigos com sentimento salvos em all_articles_with_sentiment.json ({len(all_articles)} artigos)")

print(f"
{'='*60}")
print("Pipeline multi-source testado com sucesso!")
print(f"{'='*60}")